In [1]:
from torchvision import datasets, transforms
import torch
from torch.utils.data import DataLoader


data_path_str = "./data"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cudnn.deterministic=True

transform = transforms.Compose([
    transforms.ToTensor(),
    # normalize by training set mean and standard deviation
    # resulting data has mean=0 and std=1
    transforms.Normalize((0.1307,), (0.3081,))
])

train_dataset = datasets.MNIST(data_path_str, train=True, download=True, transform=transform)
test_loader = DataLoader(
    datasets.MNIST(data_path_str, train=False, download=False, transform=transform),
    # decrease batch size if running into memory issues when testing
    # a bespoke generator is passed to avoid reproducibility issues
    shuffle=False, drop_last=False, batch_size=10000, generator=torch.Generator())

In [2]:
import torch.nn as nn
import torch.nn.functional as F


class MnistCnn(nn.Module):
    def __init__(self):
        super(MnistCnn, self).__init__()

        self.conv1 = nn.Conv2d(1, 32, 3, 1)
        self.conv2 = nn.Conv2d(32, 64, 3, 1)
        self.dropout1 = nn.Dropout(0.25)
        self.dropout2 = nn.Dropout(0.5)
        self.fc1 = nn.Linear(9216, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = self.conv1(x)
        x = F.relu(x)
        x = self.conv2(x)
        x = F.relu(x)
        x = F.max_pool2d(x, 2)
        x = self.dropout1(x)
        x = torch.flatten(x, 1)
        x = self.fc1(x)
        x = F.relu(x)
        x = self.dropout2(x)
        x = self.fc2(x)
        output = F.log_softmax(x, dim=1)

        return output

In [3]:
import numpy as np
import random


def set_seed(seed: int) -> None:
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)

In [4]:
from torch.optim import Optimizer


def train_epoch(model: torch.nn.Module, loader: DataLoader, optimizer: Optimizer) -> None:
    model.train()

    for data, target in loader:
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        output = model(data)
        loss = F.nll_loss(output, target)
        loss.backward()
        optimizer.step()

In [5]:
import numpy.random as npr
from torch.utils.data import Subset
from typing import cast


def split(nr_clients: int, iid: bool, seed: int) -> list[Subset]:
    rng = npr.default_rng(seed)

    if iid:
        splits = np.array_split(rng.permutation(len(train_dataset)), nr_clients)
    else:
        sorted_indices = np.argsort(np.array([target for _data, target in train_dataset]))
        shards = np.array_split(sorted_indices, 2 * nr_clients)
        shuffled_shard_indices = rng.permutation(len(shards))
        splits = [
            np.concatenate([shards[i] for i in inds], dtype=np.int64)
            for inds in shuffled_shard_indices.reshape(-1, 2)]

    return [Subset(train_dataset, split) for split in cast(list[list[int]], splits)]

In [6]:
from dataclasses import dataclass, field, asdict
from pandas import DataFrame


@dataclass
class RunResult:
    wall_time_by_epoch: list[float] = field(default_factory=list)
    nr_messages_by_epoch: list[int] = field(default_factory=list)
    test_accuracy_by_epoch: list[float] = field(default_factory=list)

    def as_df(self) -> DataFrame:
        return DataFrame(asdict(self))

In [7]:
from abc import ABC, abstractmethod
from torch.optim import SGD


class Client(ABC):
    def __init__(self, client_data: Subset, batch_size: int) -> None:
        self.model = MnistCnn().to(device)
        self.loader_train = DataLoader(
            client_data, batch_size=batch_size, shuffle=True, drop_last=False)

    @abstractmethod
    def update(self, weights: list[torch.Tensor], seed: int) -> list[torch.Tensor]:
        ...

In [8]:
class Server(ABC):
    def __init__(self, seed: int) -> None:
        self.clients: list[Client]
        self.name: str
        self.seed = seed
        set_seed(seed)
        self.model = MnistCnn().to(device)


    @abstractmethod
    def run(self, nr_rounds: int, seed: int) -> RunResult:
        ...


    def test(self) -> float:
        correct = 0
        self.model.eval()

        with torch.no_grad():
            for data, target in test_loader:
                data, target = data.to(device), target.to(device)
                output = self.model(data)
                pred = output.argmax(dim=1, keepdim=True)
                correct += pred.eq(target.view_as(pred)).sum().item()

        return 100. * correct / len(cast(datasets.MNIST, test_loader.dataset))

In [9]:
from time import perf_counter
from tqdm import tqdm


class CentralizedServer(Server):
    def __init__(self, lr: float, batch_size: int, seed: int) -> None:
        super().__init__(seed)
        self.optimizer = SGD(params=self.model.parameters(), lr=lr)
        self.loader_train = DataLoader(
            train_dataset, batch_size=batch_size, shuffle=True, drop_last=False)
        self.clients = []
        self.name = f"Centralized (\N{GREEK SMALL LETTER ETA}={lr}, B={batch_size})"

    def run(self, nr_rounds: int) -> RunResult:
        run_result = RunResult()

        for epoch in tqdm(range(nr_rounds), desc="Epochs", leave=False):
            set_seed(self.seed + epoch + 1)
            start_time = perf_counter()
            train_epoch(self.model, self.loader_train, self.optimizer)
            end_time = perf_counter() - start_time
            run_result.wall_time_by_epoch.append(end_time)
            run_result.nr_messages_by_epoch.append(0)
            run_result.test_accuracy_by_epoch.append(self.test())

        return run_result

In [10]:
# centralized_server = CentralizedServer(0.5, 1024, 42)
# result_centralized = centralized_server.run(2)
# result_centralized.test_accuracy_by_epoch

In [11]:
class GradientClient(Client):
    def __init__(self, client_data: Subset) -> None:
        super().__init__(client_data, len(client_data))

    def update(self, weights: list[torch.Tensor], seed: int) -> list[torch.Tensor]:
        with torch.no_grad():
            for client_values, server_values in zip(self.model.parameters(), weights):
                client_values[:] = server_values
                client_values.grad = None

        set_seed(seed)
        self.model.train()

        # this will always have one iteratioon
        for data, target in self.loader_train:
            data, target = data.to(device), target.to(device)
            output = self.model(data)
            loss = F.nll_loss(output, target)
            loss.backward()

        return [
            cast(torch.Tensor, x.grad).detach().cpu().clone()
            for x in self.model.parameters()]

In [12]:
class FedSgdGradientServer(Server):
    def __init__(
            self, lr: float,
            client_subsets: list[Subset], client_fraction: float, seed: int) -> None:
        super().__init__(seed)
        self.optimizer = SGD(params=self.model.parameters(), lr=lr)
        self.clients = [GradientClient(subset) for subset in client_subsets]
        self.nr_clients = len(client_subsets)
        self.client_sample_counts = [len(subset) for subset in client_subsets]
        self.nr_clients_per_round = max(1, round(client_fraction * self.nr_clients))
        self.rng = npr.default_rng(seed)
        self.name = f"FedSGDGradient (\N{GREEK SMALL LETTER ETA}={lr}, K={self.nr_clients}, C={client_fraction})"

    def run(self, nr_rounds: int) -> RunResult:
        run_result = RunResult()

        for nr_round in tqdm(range(nr_rounds), desc="Rounds", leave=False):
            self.model.train()
            self.optimizer.zero_grad()
            weights = [x.detach().cpu().clone() for x in self.model.parameters()]
            indices_chosen_clients = self.rng.choice(self.nr_clients, self.nr_clients_per_round, replace=False)
            chosen_sum_nr_samples = sum(self.client_sample_counts[i] for i in indices_chosen_clients)
            chosen_weighted_gradients: list[list[torch.Tensor]] = []
            round_time = 0.

            for c_i in indices_chosen_clients:
                ind = int(c_i)
                client_round_seed = self.seed + ind + 1 + nr_round * self.nr_clients_per_round
                client_start_time = perf_counter()
                client_gradients = self.clients[ind].update(weights, client_round_seed)
                client_end_time = perf_counter()
                chosen_weighted_gradients.append([
                    tens * self.client_sample_counts[ind] / chosen_sum_nr_samples
                     for tens in client_gradients])
                round_time = max(round_time, client_end_time - client_start_time)

            start_time = perf_counter()
            averaged_chosen_gradients: list[torch.Tensor] = [sum(x) for x in zip(*chosen_weighted_gradients)]

            with torch.no_grad():
                for client_gradient, server_parameter in zip(averaged_chosen_gradients, self.model.parameters()):
                    server_parameter.grad = client_gradient.to(device=device)

            self.optimizer.step()
            end_time = perf_counter()
            round_time += end_time - start_time
            run_result.wall_time_by_epoch.append(round_time)
            run_result.nr_messages_by_epoch.append(0)
            run_result.test_accuracy_by_epoch.append(self.test())

        return run_result

In [13]:
# fedsgd_gradient_server = FedSgdGradientServer(0.01, split(10, True, 42), 0.5, 42)
# result_fedsgd_gradient = fedsgd_gradient_server.run(25)
# result_fedsgd_gradient.test_accuracy_by_epoch

In [14]:
class WeightClient(Client):
    def __init__(self, client_data: Subset, lr: float, batch_size: int, nr_epochs: int) -> None:
        super().__init__(client_data, batch_size)
        self.optimizer = SGD(params=self.model.parameters(), lr=lr)
        self.nr_epochs = nr_epochs


    def update(self, weights: list[torch.Tensor], seed: int) -> list[torch.Tensor]:
        with torch.no_grad():
            for client_values, server_values in zip(self.model.parameters(), weights):
                client_values[:] = server_values

        set_seed(seed)

        for _epoch in range(self.nr_epochs):
            train_epoch(self.model, self.loader_train, self.optimizer)

        return [x.detach().cpu().clone() for x in self.model.parameters()]

In [15]:
class FedAvgServer(Server):
    def __init__(
            self, lr: float, batch_size: int, client_subsets: list[Subset],
            client_fraction: float, nr_local_epochs: int, seed: int) -> None:
        super().__init__(seed)
        self.clients = [
            WeightClient(subset, lr, batch_size, nr_local_epochs)
            for subset in client_subsets]
        self.nr_clients = len(client_subsets)
        self.client_sample_counts = [len(subset) for subset in client_subsets]
        self.nr_clients_per_round = max(1, round(client_fraction * self.nr_clients))
        self.rng = npr.default_rng(seed)
        self.name = f"FedAvg (\N{GREEK SMALL LETTER ETA}={lr}, K={self.nr_clients}, C={client_fraction}, B={batch_size}, E={nr_local_epochs})"

    def run(self, nr_rounds: int) -> RunResult:
        run_result = RunResult()

        for nr_round in tqdm(range(nr_rounds), desc="Rounds", leave=False):
            self.model.train()
            weights = [x.detach().cpu().clone() for x in self.model.parameters()]
            indices_chosen_clients = self.rng.choice(self.nr_clients, self.nr_clients_per_round, replace=False)
            chosen_sum_nr_samples = sum(self.client_sample_counts[i] for i in indices_chosen_clients)
            chosen_weighted_weights: list[list[torch.Tensor]] = []
            round_time = 0.

            for c_i in indices_chosen_clients:
                ind = int(c_i)
                client_round_seed = self.seed + ind + 1 + nr_round * self.nr_clients_per_round
                client_start_time = perf_counter()
                client_weights = self.clients[ind].update(weights, client_round_seed)
                client_end_time = perf_counter()
                chosen_weighted_weights.append([
                    self.client_sample_counts[ind] / chosen_sum_nr_samples * tens
                     for tens in client_weights])
                round_time = max(round_time, client_end_time - client_start_time)

            start_time = perf_counter()
            averaged_chosen_weights: list[torch.Tensor] = [sum(x) for x in zip(*chosen_weighted_weights)]

            with torch.no_grad():
                for client_weight, server_parameter in zip(averaged_chosen_weights, self.model.parameters()):
                    server_parameter[:] = client_weight.to(device=device)

            end_time = perf_counter()
            round_time += end_time - start_time
            run_result.wall_time_by_epoch.append(round_time)
            run_result.nr_messages_by_epoch.append(0)
            run_result.test_accuracy_by_epoch.append(self.test())

        return run_result

In [16]:
# fedavg_server = FedAvgServer(0.01, 100, split(100, False, 42), 0.1, 10, 42)
# result_fedavg = fedavg_server.run(10)
# result_fedavg.test_accuracy_by_epoch

In [17]:
class FedSgdWeightServer(FedAvgServer):
    def __init__(
            self, lr: float, client_subsets: list[Subset],
            client_fraction: float, seed: int) -> None:
        super().__init__(lr, len(train_dataset), client_subsets, client_fraction, 1, seed)
        self.name = f"FedSGDGradient (\N{GREEK SMALL LETTER ETA}={lr}, K={self.nr_clients}, C={client_fraction})"

In [18]:
# fedsgd_weight_server = FedSgdWeightServer(0.01, split(10, True, 42), 0.5, 42)
# result_fedsgd_weight = fedsgd_weight_server.run(25)
# result_fedsgd_weight.test_accuracy_by_epoch